#Set up

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Função do gráfico

In [ ]:
def grafico_card_indicador_ultimo_mes(
    df_plot,
    coluna_valor="qtd_ativos",
    coluna_competencia="competencia",
    coluna_pop="pop",
    coluna_yoy="yoy",
    coluna_meta=None,
    coluna_benchmark=None,
    titulo="Indicador - última competência",
    subtitulo=None,
    detalhe="completo",
    maior_melhor=True,
    mostrar_competencia=True,
    mostrar_pop=True,
    mostrar_yoy=True,
    mostrar_meta=True,
    mostrar_benchmark=True,
    mostrar_comparacao_meta=True,
    mostrar_comparacao_benchmark=True,
    mostrar_valor_meta_benchmark=False,
    casas_decimais_valor=0,
    casas_decimais_percentual=1,
    prefixo_valor="",
    sufixo_valor="",
    texto_competencia_prefixo="Competência:",
    texto_pop="PoP",
    texto_yoy="YoY",
    texto_meta="Meta",
    texto_benchmark="Benchmark",
    width=950,
    height=600,
    template="plotly_white",
    cor_titulo="#173A70",
    cor_subtitulo="#4B5E7A",
    cor_valor="#1F3B64",
    cor_texto="#334155",
    cor_positivo="#16A34A",
    cor_negativo="#DC2626",
    cor_neutro="#64748B",
    tamanho_titulo=22,
    tamanho_subtitulo=15,
    tamanho_competencia=16,
    tamanho_valor=58,
    tamanho_variacoes=18,
    tamanho_referencias=16,
    pos_y_titulo=0.93,
    pos_y_subtitulo=0.86,
    pos_y_competencia=0.75,
    pos_y_valor=0.52,
    pos_y_variacoes=0.31,
    pos_y_referencias=0.15,
    margem_esquerda=40,
    margem_direita=40,
    margem_superior=40,
    margem_inferior=40,
):
    """
    Cria um card executivo com:
    - título
    - subtítulo opcional
    - competência
    - valor principal
    - variações PoP e YoY
    - comparação com meta e benchmark

    Observação:
    A função usa a primeira linha de `df_plot` como base para o card.
    """

    # ------------------------------------------------------------
    # 1) Validação do parâmetro de detalhe
    # ------------------------------------------------------------
    detalhes_validos = {"completo", "pop", "yoy", "nenhum"}
    if detalhe not in detalhes_validos:
        raise ValueError(
            f"Parâmetro 'detalhe' inválido. Use um destes: {sorted(detalhes_validos)}"
        )

    # ------------------------------------------------------------
    # 2) Validação do dataframe
    # ------------------------------------------------------------
    if df_plot is None or df_plot.empty:
        raise ValueError("O dataframe `df_plot` está vazio.")

    if coluna_valor not in df_plot.columns:
        raise ValueError(f"A coluna '{coluna_valor}' não existe no dataframe.")

    if mostrar_competencia and coluna_competencia not in df_plot.columns:
        raise ValueError(f"A coluna '{coluna_competencia}' não existe no dataframe.")

    row = df_plot.iloc[0]

    valor = row[coluna_valor]
    competencia = row[coluna_competencia] if coluna_competencia in df_plot.columns else None
    pop = row[coluna_pop] if coluna_pop in df_plot.columns else np.nan
    yoy = row[coluna_yoy] if coluna_yoy in df_plot.columns else np.nan
    meta = row[coluna_meta] if coluna_meta and coluna_meta in df_plot.columns else np.nan
    benchmark = row[coluna_benchmark] if coluna_benchmark and coluna_benchmark in df_plot.columns else np.nan

    # ------------------------------------------------------------
    # 3) Funções auxiliares
    # ------------------------------------------------------------
    def formatar_numero(valor, casas=0, prefixo="", sufixo=""):
        """Formata número em padrão brasileiro."""
        if pd.isna(valor):
            return "NA"
        texto = f"{valor:,.{casas}f}"
        texto = texto.replace(",", "X").replace(".", ",").replace("X", ".")
        return f"{prefixo}{texto}{sufixo}"

    def formatar_percentual(valor):
        """Formata percentual com sinal opcional."""
        if pd.isna(valor):
            return None
        sinal = "+" if valor > 0 else ""
        texto = f"{sinal}{abs(valor):.{casas_decimais_percentual}f}%"
        return texto.replace(".", ",")

    def definir_cor_resultado(delta):
        """
        Define a cor conforme a regra de negócio:
        - maior_melhor=True: acima é bom
        - maior_melhor=False: acima é ruim
        """
        if pd.isna(delta) or delta == 0:
            return cor_neutro

        if maior_melhor:
            return cor_positivo if delta > 0 else cor_negativo
        return cor_negativo if delta > 0 else cor_positivo

    def definir_seta(delta):
        """Define a seta visual do movimento."""
        if pd.isna(delta) or delta == 0:
            return "•"
        return "▲" if delta > 0 else "▼"

    def montar_linha_variacao(rotulo, variacao):
        """Monta a linha de PoP ou YoY."""
        if pd.isna(variacao):
            return None

        cor = definir_cor_resultado(variacao)
        seta = definir_seta(variacao)
        texto_pct = f"{abs(variacao):.{casas_decimais_percentual}f}%".replace(".", ",")
        texto = f"{rotulo} {seta} {texto_pct}"
        return f"<span style='color:{cor};'><b>{texto}</b></span>"

    def montar_linha_comparacao(nome_ref, valor_atual, valor_ref):
        """
        Exemplo:
        Meta 12.500 | Δ +340 | Δ% +2,7%
        """
        if pd.isna(valor_ref):
            return None

        delta_abs = valor_atual - valor_ref
        delta_pct = np.nan if valor_ref == 0 else (delta_abs / valor_ref) * 100
        cor = definir_cor_resultado(delta_abs)

        texto_ref = formatar_numero(
            valor_ref,
            casas=casas_decimais_valor,
            prefixo=prefixo_valor,
            sufixo=sufixo_valor
        )

        sinal_abs = "+" if delta_abs > 0 else "-"
        texto_delta_abs = formatar_numero(
            abs(delta_abs),
            casas=casas_decimais_valor,
            prefixo=prefixo_valor,
            sufixo=sufixo_valor
        )

        if pd.isna(delta_pct):
            texto_delta_pct = "NA"
        else:
            sinal_pct = "+" if delta_pct > 0 else "-"
            texto_delta_pct = f"{sinal_pct}{abs(delta_pct):.{casas_decimais_percentual}f}%".replace(".", ",")

        texto = (
            f"<b>{nome_ref}</b>: {texto_ref} "
            f"| Δ {sinal_abs}{texto_delta_abs} "
            f"| Δ% {texto_delta_pct}"
        )
        return f"<span style='color:{cor};'>{texto}</span>"

    # ------------------------------------------------------------
    # 4) Montagem dos blocos de texto
    # ------------------------------------------------------------
    valor_formatado = formatar_numero(
        valor,
        casas=casas_decimais_valor,
        prefixo=prefixo_valor,
        sufixo=sufixo_valor
    )

    # Bloco de variações
    linhas_variacoes = []

    if detalhe in {"completo", "pop"} and mostrar_pop:
        linha = montar_linha_variacao(texto_pop, pop)
        if linha:
            linhas_variacoes.append(linha)

    if detalhe in {"completo", "yoy"} and mostrar_yoy:
        linha = montar_linha_variacao(texto_yoy, yoy)
        if linha:
            linhas_variacoes.append(linha)

    texto_variacoes = "<br>".join(linhas_variacoes)

    # Bloco de referências
    linhas_referencias = []

    if mostrar_valor_meta_benchmark:
        if mostrar_meta and not pd.isna(meta):
            linhas_referencias.append(
                f"<span style='color:{cor_texto};'><b>{texto_meta}</b>: "
                f"{formatar_numero(meta, casas_decimais_valor, prefixo_valor, sufixo_valor)}</span>"
            )

        if mostrar_benchmark and not pd.isna(benchmark):
            linhas_referencias.append(
                f"<span style='color:{cor_texto};'><b>{texto_benchmark}</b>: "
                f"{formatar_numero(benchmark, casas_decimais_valor, prefixo_valor, sufixo_valor)}</span>"
            )

    if mostrar_comparacao_meta and not pd.isna(meta):
        linha = montar_linha_comparacao(texto_meta, valor, meta)
        if linha:
            linhas_referencias.append(linha)

    if mostrar_comparacao_benchmark and not pd.isna(benchmark):
        linha = montar_linha_comparacao(texto_benchmark, valor, benchmark)
        if linha:
            linhas_referencias.append(linha)

    texto_referencias = "<br>".join(linhas_referencias)

    # ------------------------------------------------------------
    # 5) Criação da figura
    # ------------------------------------------------------------
    fig = go.Figure()

    fig.update_layout(
        template=template,
        width=width,
        height=height,
        margin=dict(
            l=margem_esquerda,
            r=margem_direita,
            t=margem_superior,
            b=margem_inferior
        ),
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        showlegend=False,
    )

    # ------------------------------------------------------------
    # 6) Título
    # ------------------------------------------------------------
    fig.add_annotation(
        x=0.5,
        y=pos_y_titulo,
        xref="paper",
        yref="paper",
        text=titulo,
        showarrow=False,
        font=dict(size=tamanho_titulo, color=cor_titulo),
        align="center"
    )

    # ------------------------------------------------------------
    # 7) Subtítulo
    # ------------------------------------------------------------
    if subtitulo:
        fig.add_annotation(
            x=0.5,
            y=pos_y_subtitulo,
            xref="paper",
            yref="paper",
            text=subtitulo,
            showarrow=False,
            font=dict(size=tamanho_subtitulo, color=cor_subtitulo),
            align="center"
        )

    # ------------------------------------------------------------
    # 8) Competência
    # ------------------------------------------------------------
    if mostrar_competencia and competencia is not None:
        fig.add_annotation(
            x=0.5,
            y=pos_y_competencia,
            xref="paper",
            yref="paper",
            text=f"{texto_competencia_prefixo} {competencia}",
            showarrow=False,
            font=dict(size=tamanho_competencia, color=cor_titulo),
            align="center"
        )

    # ------------------------------------------------------------
    # 9) Valor principal
    # ------------------------------------------------------------
    fig.add_annotation(
        x=0.5,
        y=pos_y_valor,
        xref="paper",
        yref="paper",
        text=f"<b>{valor_formatado}</b>",
        showarrow=False,
        font=dict(size=tamanho_valor, color=cor_valor),
        align="center"
    )

    # ------------------------------------------------------------
    # 10) Variações
    # ------------------------------------------------------------
    if detalhe != "nenhum" and texto_variacoes:
        fig.add_annotation(
            x=0.5,
            y=pos_y_variacoes,
            xref="paper",
            yref="paper",
            text=texto_variacoes,
            showarrow=False,
            font=dict(size=tamanho_variacoes),
            align="center"
        )

    # ------------------------------------------------------------
    # 11) Meta e benchmark
    # ------------------------------------------------------------
    if texto_referencias:
        fig.add_annotation(
            x=0.5,
            y=pos_y_referencias,
            xref="paper",
            yref="paper",
            text=texto_referencias,
            showarrow=False,
            font=dict(size=tamanho_referencias),
            align="center"
        )

    return fig

# Dados de demostração

In [ ]:
df_card = pd.DataFrame({
    "competencia": ["2026-02"],
    "qtd_ativos": [12840],
    "pop": [2.7],
    "yoy": [-1.8],
    "meta": [12500],
    "benchmark": [13000]
})

# Plotagem

In [ ]:
fig = grafico_card_indicador_ultimo_mes(
    df_plot=df_card,
    coluna_valor="qtd_ativos",
    coluna_competencia="competencia",
    coluna_pop="pop",
    coluna_yoy="yoy",
    coluna_meta="meta",
    coluna_benchmark="benchmark",
    titulo="Total de beneficiários ativos - última competência",
    subtitulo="Indicador consolidado da base assistencial",
    detalhe="completo",
    maior_melhor=True,
    mostrar_meta=True,
    mostrar_benchmark=True,
    mostrar_comparacao_meta=True,
    mostrar_comparacao_benchmark=True,
    casas_decimais_valor=0,
    casas_decimais_percentual=1,
    width=1000,
    height=560
)

fig.show()